# AutoShot video to GCS keyframes

This notebook extracts AutoShot keyframes from the Kaggle-mounted AI Challenge 2025 videos and uploads them to Google Cloud Storage.

Expected Kaggle formats are discovered recursively, including paths such as:

```text
/kaggle/input/ai-challenge-2025/Videos/Videos/Videos_L21_a/video/L21_V001.mp4
/kaggle/input/.../ai-challenge-2025/Videos/Videos/Video_L30_a/video/L30_V001.mp4
```

Outputs follow the same shape as `scripts/loaders/video_to_frame_gcs.py`: keyframes under `processed/keyframes/...` and run artifacts under `processed/keyframes_manifests/...`.

In [ ]:
from pathlib import Path

FORCE_REWRITE_HELPERS = False
PARAMS_PATH = Path("video_to_frame_gcs_params.py")
PIPELINE_PATH = Path("video_to_frame_gcs_kaggle.py")

PARAMS_TEMPLATE = '"""Parameters for the Kaggle AutoShot -> GCS keyframe notebook.\n\nEdit this file first when running locally, or edit the generated copy at\n``/kaggle/working/video_to_frame_gcs_params.py`` when running on Kaggle.\nDo not put service-account JSON content in this file; use Kaggle Secrets or\nenvironment variables instead.\n"""\n\n# Kaggle input.\nINPUT_ROOT = ""  # Empty = auto-detect /kaggle/input/ai-challenge-2025.\nEXPECTED_BATCHES = [\n    "L21",\n    "L22",\n    "L23",\n    "L24",\n    "L25",\n    "L26",\n    "L27",\n    "L28",\n    "L29",\n    "L30",\n]\nBATCH_REGEX = r"(?i)(?:^|[/_\\\\-])(?:videos?_)?([A-Z]\\d{2})(?:[_/\\\\-]|$)"\nVIDEO_EXTENSIONS = [".mp4", ".avi", ".mov", ".mkv", ".webm"]\n\n# Dataset/model metadata used in GCS paths and CSV rows.\nDATASET_ID = "ai_challenge_2025"\nSOURCE_VERSION = "kaggle_current"\nPROFILE_VERSION = "autoshot_v1"\nRAW_PREFIX = "raw/source=kaggle"\nRAW_RELATIVE_PATH_PREFIX = "ai-challenge-2025"  # Set "" if raw GCS objects omit this folder.\nKEYFRAMES_PREFIX = "processed/keyframes"\nMANIFESTS_PREFIX = "processed/keyframes_manifests"\nRAW_VIDEO_URI_MODE = "gcs_expected"  # "gcs_expected" or "kaggle".\n\n# GCS credentials. Empty values are resolved from env vars or Kaggle Secrets.\nGCS_BUCKET = ""\nGCS_BUCKET_SECRET_NAME = "GCS_BUCKET"\nGCS_CREDENTIALS_FILE = ""\nGCS_CREDENTIALS_JSON_SECRET_NAME = "GCS_CREDENTIALS_JSON"\n\n# AutoShot runtime.\nAUTOSHOT_REPO_DIR = "/kaggle/working/AutoShot"\nAUTOSHOT_REPO_URL = "https://github.com/wentaozhu/AutoShot.git"\nAUTO_CLONE_AUTOSHOT = True\nCHECKPOINT_PATH = "/kaggle/input/models/khngxuninh/autoshot/pytorch/default/1/ckpt_0_200_0.pth"\nDEVICE = "auto"  # "auto", "cpu", "cuda", or "cuda:0".\nTHRESHOLD = 0.296\nMIN_SHOT_LEN = 5\nJPEG_QUALITY = 95\n\n# Local Kaggle output.\nRUN_DIR = "/kaggle/working/frame_extraction_runs"\nSCRATCH_DIR = "/kaggle/working/autoshot_scratch"\nCLEANUP_LOCAL_IMAGES_AFTER_UPLOAD = True\n\n# Upload behavior.\nUPLOAD_TO_GCS = True\nUPLOAD_RUN_ARTIFACTS = True\nSKIP_EXISTING = True\nOVERWRITE = False\n\n# Progress/logging.\nUSE_TQDM = True\nVERBOSE = False\nLOG_EVERY_VIDEO = True\n\n# Notebook cells.\nDRY_RUN_BATCHES = "all"\nDRY_RUN_MAX_VIDEOS = 20\nDEMO_BATCHES = "L21"\nDEMO_MAX_VIDEOS = 2\nFULL_BATCHES = "all"\nFULL_MAX_VIDEOS = None\n\n# Safety guard for the final full-run notebook cell.\nCONFIRM_FULL_RUN = ""  # Set to "RUN_FULL_DATASET" before executing full run.\n'
PIPELINE_TEMPLATE = '"""Kaggle AutoShot keyframe extraction and GCS upload helper.\n\nThis module is designed for ``video_to_frame_gcs.ipynb``. It extracts\nfirst/middle/last representative frames per AutoShot shot from Kaggle-mounted\nvideos, uploads images to GCS, and writes manifest artifacts compatible with\n``scripts/loaders/video_to_frame_gcs.py`` and the backend data model.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport importlib\nimport importlib.util\nimport json\nimport logging\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport uuid\nfrom dataclasses import dataclass\nfrom datetime import UTC, datetime\nfrom pathlib import Path, PurePosixPath\nfrom typing import Any, Iterable\n\n\nDEFAULT_VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}\n\nSHOT_SEGMENTS_COLUMNS = [\n    "dataset_id",\n    "batch_id",\n    "video_id",\n    "video_name",\n    "video_gcs_uri",\n    "video_gcs_generation",\n    "shot_id",\n    "shot_id_local",\n    "shot_start_frame",\n    "shot_end_frame",\n    "shot_start_sec",\n    "shot_end_sec",\n    "frame_type",\n    "frame_idx",\n    "frame_sec",\n    "keyframe_id",\n    "image_rel_path",\n    "image_gcs_uri",\n    "image_storage_key",\n    "boundary_threshold",\n    "min_shot_len",\n    "saved",\n    "fps",\n    "total_frames_opencv",\n    "profile_version",\n    "run_id",\n]\n\n\n@dataclass(frozen=True)\nclass GcsUri:\n    bucket: str\n    blob_name: str\n\n    @property\n    def uri(self) -> str:\n        return f"gs://{self.bucket}/{self.blob_name}"\n\n\n@dataclass(frozen=True)\nclass RunLayout:\n    run_id: str\n    run_dir: Path\n    frames_dir: Path\n    artifacts_dir: Path\n    manifest_path: Path\n    shot_segments_path: Path\n    errors_path: Path\n    video_summaries_path: Path\n    summary_path: Path\n    log_path: Path\n    gcs_artifact_prefix: str\n\n\ndef cfg_value(cfg: Any, name: str, default: Any = None) -> Any:\n    return getattr(cfg, name, default)\n\n\ndef utc_now_iso() -> str:\n    return datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")\n\n\ndef new_run_id(prefix: str) -> str:\n    stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")\n    safe_prefix = re.sub(r"[^A-Za-z0-9_.-]+", "_", prefix).strip("_").lower()\n    return f"{safe_prefix}_{stamp}_{uuid.uuid4().hex[:8]}"\n\n\ndef normalize_prefix(prefix: str) -> str:\n    return str(prefix or "").strip().strip("/")\n\n\ndef parse_gcs_uri(value: str) -> GcsUri:\n    if not value.startswith("gs://"):\n        raise ValueError(f"Expected gs:// URI, got: {value}")\n    bucket, sep, blob_name = value[len("gs://") :].partition("/")\n    if not bucket or not sep or not blob_name:\n        raise ValueError(f"Invalid GCS URI: {value}")\n    return GcsUri(bucket=bucket, blob_name=blob_name)\n\n\ndef read_kaggle_secret(name: str) -> str:\n    if not name:\n        return ""\n    try:\n        from kaggle_secrets import UserSecretsClient\n\n        return UserSecretsClient().get_secret(name) or ""\n    except Exception:\n        return ""\n\n\ndef resolve_bucket_name(cfg: Any, require: bool) -> str:\n    configured = str(cfg_value(cfg, "GCS_BUCKET", "") or "").strip()\n    env_value = os.environ.get("GCS_BUCKET", "").strip()\n    secret_name = str(cfg_value(cfg, "GCS_BUCKET_SECRET_NAME", "GCS_BUCKET") or "").strip()\n    secret_value = read_kaggle_secret(secret_name).strip()\n    raw_value = configured or env_value or secret_value\n    if not raw_value:\n        if require:\n            raise RuntimeError("Set GCS_BUCKET in params, env vars, or Kaggle Secrets.")\n        return ""\n    if raw_value.startswith("gs://"):\n        return parse_gcs_uri(raw_value.rstrip("/") + "/_").bucket\n    return raw_value.strip().strip("/")\n\n\ndef make_storage_client(cfg: Any):\n    try:\n        from google.cloud import storage\n    except ImportError as exc:\n        raise RuntimeError("Install google-cloud-storage first.") from exc\n\n    credentials_file = str(cfg_value(cfg, "GCS_CREDENTIALS_FILE", "") or os.environ.get("GCS_CREDENTIALS_FILE", "")).strip()\n    credentials_json = os.environ.get("GCS_CREDENTIALS_JSON", "").strip()\n    secret_name = str(cfg_value(cfg, "GCS_CREDENTIALS_JSON_SECRET_NAME", "GCS_CREDENTIALS_JSON") or "").strip()\n    credentials_json = credentials_json or read_kaggle_secret(secret_name).strip()\n\n    if credentials_json:\n        from google.oauth2 import service_account\n\n        credentials = service_account.Credentials.from_service_account_info(json.loads(credentials_json))\n        return storage.Client(project=credentials.project_id, credentials=credentials)\n    if credentials_file:\n        return storage.Client.from_service_account_json(credentials_file)\n    return storage.Client()\n\n\ndef build_raw_video_uri(cfg: Any, bucket_name: str, batch_id: str, relative_path: str) -> str:\n    mode = str(cfg_value(cfg, "RAW_VIDEO_URI_MODE", "gcs_expected")).strip().lower()\n    if mode == "kaggle" or not bucket_name:\n        return f"kaggle://{relative_path.lstrip(\'/\')}"\n    raw_prefix = normalize_prefix(cfg_value(cfg, "RAW_PREFIX", "raw/source=kaggle"))\n    raw_relative_prefix = normalize_prefix(cfg_value(cfg, "RAW_RELATIVE_PATH_PREFIX", "ai-challenge-2025"))\n    raw_relative_path = relative_path.strip("/")\n    if raw_relative_prefix and not raw_relative_path.lower().startswith(raw_relative_prefix.lower() + "/"):\n        raw_relative_path = f"{raw_relative_prefix}/{raw_relative_path}"\n    dataset_id = str(cfg_value(cfg, "DATASET_ID", "ai_challenge_2025"))\n    source_version = str(cfg_value(cfg, "SOURCE_VERSION", "kaggle_current"))\n    object_key = "/".join(\n        [\n            raw_prefix,\n            f"dataset={dataset_id}",\n            f"source_version={source_version}",\n            f"batch={batch_id}",\n            raw_relative_path,\n        ]\n    )\n    return f"gs://{bucket_name}/{object_key}"\n\n\ndef build_keyframe_prefix(cfg: Any, dataset_id: str, batch_id: str, profile_version: str, video_id: str) -> str:\n    return (\n        f"{normalize_prefix(cfg_value(cfg, \'KEYFRAMES_PREFIX\', \'processed/keyframes\'))}/"\n        f"dataset={dataset_id}/"\n        f"batch={batch_id}/"\n        f"profile={profile_version}/"\n        f"video_id={video_id}/"\n    )\n\n\ndef build_artifact_prefix(cfg: Any, dataset_id: str, batch_id: str, profile_version: str, run_id: str) -> str:\n    return (\n        f"{normalize_prefix(cfg_value(cfg, \'MANIFESTS_PREFIX\', \'processed/keyframes_manifests\'))}/"\n        f"dataset={dataset_id}/"\n        f"batch={batch_id}/"\n        f"profile={profile_version}/"\n        f"run_id={run_id}/"\n    )\n\n\ndef make_run_layout(cfg: Any, run_id: str, artifact_batch_id: str) -> RunLayout:\n    run_dir = Path(str(cfg_value(cfg, "RUN_DIR", "/kaggle/working/frame_extraction_runs"))) / run_id\n    artifacts_dir = run_dir / "artifacts"\n    frames_dir = run_dir / "frames"\n    artifacts_dir.mkdir(parents=True, exist_ok=True)\n    frames_dir.mkdir(parents=True, exist_ok=True)\n\n    dataset_id = str(cfg_value(cfg, "DATASET_ID", "ai_challenge_2025"))\n    profile_version = str(cfg_value(cfg, "PROFILE_VERSION", "autoshot_v1"))\n    gcs_artifact_prefix = build_artifact_prefix(cfg, dataset_id, artifact_batch_id, profile_version, run_id)\n\n    return RunLayout(\n        run_id=run_id,\n        run_dir=run_dir,\n        frames_dir=frames_dir,\n        artifacts_dir=artifacts_dir,\n        manifest_path=artifacts_dir / "processing_manifest.jsonl",\n        shot_segments_path=artifacts_dir / "shot_segments.csv",\n        errors_path=artifacts_dir / "errors.jsonl",\n        video_summaries_path=artifacts_dir / "video_summaries.jsonl",\n        summary_path=artifacts_dir / "summary.json",\n        log_path=run_dir / "run.log",\n        gcs_artifact_prefix=gcs_artifact_prefix,\n    )\n\n\ndef setup_logging(layout: RunLayout, verbose: bool) -> logging.Logger:\n    logger = logging.getLogger("kaggle_autoshot_gcs")\n    logger.handlers.clear()\n    logger.setLevel(logging.DEBUG if verbose else logging.INFO)\n    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")\n\n    file_handler = logging.FileHandler(layout.log_path, encoding="utf-8")\n    file_handler.setFormatter(formatter)\n    file_handler.setLevel(logging.DEBUG)\n    logger.addHandler(file_handler)\n\n    console_handler = logging.StreamHandler(sys.stdout)\n    console_handler.setFormatter(formatter)\n    console_handler.setLevel(logging.DEBUG if verbose else logging.INFO)\n    logger.addHandler(console_handler)\n    return logger\n\n\ndef write_json(path: Path, payload: dict[str, Any]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, sort_keys=True), encoding="utf-8")\n\n\ndef append_jsonl(path: Path, row: dict[str, Any]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("a", encoding="utf-8") as handle:\n        handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\\n")\n\n\ndef write_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as handle:\n        for row in rows:\n            handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\\n")\n\n\ndef init_csv(path: Path, fieldnames: list[str]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fieldnames)\n        writer.writeheader()\n\n\ndef append_csv_rows(path: Path, rows: list[dict[str, Any]], fieldnames: list[str]) -> None:\n    if not rows:\n        return\n    with path.open("a", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fieldnames)\n        for row in rows:\n            writer.writerow({key: row.get(key, "") for key in fieldnames})\n\n\ndef parse_selected_batches(raw: Any, expected_batches: list[str]) -> list[str]:\n    expected = [batch.upper() for batch in expected_batches]\n    if isinstance(raw, str):\n        value = raw.strip()\n        if value.lower() in {"", "all", "*"}:\n            return expected\n        selected = [part.strip().upper() for part in value.split(",") if part.strip()]\n    else:\n        selected = [str(part).strip().upper() for part in raw if str(part).strip()]\n    unknown = sorted(set(selected) - set(expected))\n    if unknown:\n        raise ValueError(f"Unknown batch(es): {\', \'.join(unknown)}. Expected: {\', \'.join(expected)}")\n    return selected\n\n\ndef resolve_input_root(cfg: Any) -> Path:\n    configured = str(cfg_value(cfg, "INPUT_ROOT", "") or "").strip()\n    candidates: list[Path] = []\n    if configured:\n        candidates.append(Path(configured))\n    candidates.extend(\n        [\n            Path("/kaggle/input/ai-challenge-2025"),\n            Path("/kaggle/input/datasets/aresusayhi/ai-challenge-2025"),\n        ]\n    )\n    for candidate in candidates:\n        if candidate.exists() and candidate.is_dir():\n            return candidate\n\n    kaggle_input = Path("/kaggle/input")\n    if kaggle_input.exists():\n        for child in sorted(kaggle_input.iterdir()):\n            if child.is_dir() and any(child.rglob("*.mp4")):\n                return child\n\n    searched = ", ".join(str(item) for item in candidates)\n    raise FileNotFoundError(f"Cannot find Kaggle input root. Checked: {searched}")\n\n\ndef detect_batch(relative_path: str, cfg: Any) -> str | None:\n    pattern = str(cfg_value(cfg, "BATCH_REGEX", r"(?i)(?:^|[/_\\\\-])(?:videos?_)?([A-Z]\\d{2})(?:[_/\\\\-]|$)"))\n    match = re.search(pattern, relative_path)\n    if match:\n        return match.group(1).upper()\n    stem_match = re.match(r"(?i)^([A-Z]\\d{2})[_-]", PurePosixPath(relative_path).name)\n    if stem_match:\n        return stem_match.group(1).upper()\n    return None\n\n\ndef discover_videos(cfg: Any, batches: Any, max_videos: int | None, bucket_name: str) -> tuple[list[dict[str, Any]], list[dict[str, Any]], Path]:\n    input_root = resolve_input_root(cfg)\n    expected = [str(item).upper() for item in cfg_value(cfg, "EXPECTED_BATCHES", [])]\n    selected_batches = set(parse_selected_batches(batches, expected))\n    extensions = {str(item).lower() for item in cfg_value(cfg, "VIDEO_EXTENSIONS", DEFAULT_VIDEO_EXTENSIONS)}\n    dataset_id = str(cfg_value(cfg, "DATASET_ID", "ai_challenge_2025"))\n    profile_version = str(cfg_value(cfg, "PROFILE_VERSION", "autoshot_v1"))\n\n    records: list[dict[str, Any]] = []\n    unmapped: list[dict[str, Any]] = []\n\n    for path in sorted(input_root.rglob("*"), key=lambda item: item.as_posix()):\n        if not path.is_file() or path.suffix.lower() not in extensions:\n            continue\n        rel = path.relative_to(input_root).as_posix()\n        batch_id = detect_batch(rel, cfg)\n        if not batch_id or batch_id not in set(expected):\n            unmapped.append(\n                {\n                    "relative_path": rel,\n                    "local_path": str(path),\n                    "reason": "batch_not_detected_or_unexpected",\n                    "size_bytes": path.stat().st_size,\n                }\n            )\n            continue\n        if batch_id not in selected_batches:\n            continue\n\n        video_id = path.stem\n        output_prefix = build_keyframe_prefix(cfg, dataset_id, batch_id, profile_version, video_id)\n        records.append(\n            {\n                "manifest_schema_version": 1,\n                "dataset_id": dataset_id,\n                "batch_id": batch_id,\n                "source_version": str(cfg_value(cfg, "SOURCE_VERSION", "kaggle_current")),\n                "profile_version": profile_version,\n                "video_id": video_id,\n                "video_name": path.name,\n                "relative_path": rel,\n                "local_video_path": str(path),\n                "input_gcs_uri": build_raw_video_uri(cfg, bucket_name, batch_id, rel),\n                "input_generation": "",\n                "input_size_bytes": path.stat().st_size,\n                "output_bucket": bucket_name,\n                "output_prefix": output_prefix,\n                "threshold": float(cfg_value(cfg, "THRESHOLD", 0.296)),\n                "min_shot_len": int(cfg_value(cfg, "MIN_SHOT_LEN", 5)),\n                "planned_at": utc_now_iso(),\n            }\n        )\n        if max_videos is not None and len(records) >= max_videos:\n            break\n\n    return records, unmapped, input_root\n\n\ndef require_module(module_name: str, install_hint: str) -> None:\n    if importlib.util.find_spec(module_name) is None:\n        raise RuntimeError(f"Missing dependency \'{module_name}\'. Install with: {install_hint}")\n\n\ndef ensure_autoshot_repo(cfg: Any, logger: logging.Logger) -> Path:\n    repo_dir = Path(str(cfg_value(cfg, "AUTOSHOT_REPO_DIR", "/kaggle/working/AutoShot"))).expanduser()\n    required = [\n        repo_dir / "supernet_flattransf_3_8_8_8_13_12_0_16_60.py",\n        repo_dir / "utils.py",\n    ]\n    if all(path.exists() for path in required):\n        return repo_dir\n\n    if not bool(cfg_value(cfg, "AUTO_CLONE_AUTOSHOT", True)):\n        missing = ", ".join(str(path) for path in required if not path.exists())\n        raise FileNotFoundError(f"AutoShot repo is missing required file(s): {missing}")\n\n    repo_url = str(cfg_value(cfg, "AUTOSHOT_REPO_URL", "https://github.com/wentaozhu/AutoShot.git"))\n    repo_dir.parent.mkdir(parents=True, exist_ok=True)\n    if repo_dir.exists() and not any(repo_dir.iterdir()):\n        repo_dir.rmdir()\n    logger.info("cloning AutoShot repo to %s", repo_dir)\n    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)\n\n    missing_after_clone = [str(path) for path in required if not path.exists()]\n    if missing_after_clone:\n        raise FileNotFoundError(f"AutoShot clone is incomplete: {\', \'.join(missing_after_clone)}")\n    return repo_dir\n\n\ndef add_repo_to_path(repo_dir: Path) -> None:\n    resolved = str(repo_dir.resolve())\n    if resolved not in sys.path:\n        sys.path.insert(0, resolved)\n\n\ndef select_device(raw_device: str) -> str:\n    if raw_device != "auto":\n        return raw_device\n    require_module("torch", "Kaggle GPU notebooks normally include torch.")\n    import torch\n\n    return "cuda" if torch.cuda.is_available() else "cpu"\n\n\ndef load_autoshot_model(repo_dir: Path, checkpoint_path: Path, device: str, logger: logging.Logger):\n    require_module("torch", "Kaggle GPU notebooks normally include torch.")\n    import torch\n\n    add_repo_to_path(repo_dir)\n    from supernet_flattransf_3_8_8_8_13_12_0_16_60 import TransNetV2Supernet\n\n    model = TransNetV2Supernet().eval()\n    checkpoint = torch.load(str(checkpoint_path), map_location=device)\n    pretrained_state = checkpoint["net"] if isinstance(checkpoint, dict) and "net" in checkpoint else checkpoint\n    model_state = model.state_dict()\n    matched_state = {\n        key: value\n        for key, value in pretrained_state.items()\n        if key in model_state and tuple(value.shape) == tuple(model_state[key].shape)\n    }\n    if not matched_state:\n        raise RuntimeError("Checkpoint did not match any AutoShot model parameters.")\n    model_state.update(matched_state)\n    model.load_state_dict(model_state)\n    model = model.to(device).eval()\n    logger.info("loaded AutoShot checkpoint=%s matched_params=%d/%d device=%s", checkpoint_path, len(matched_state), len(model_state), device)\n    return model\n\n\ndef resolve_ffmpeg_executable() -> str:\n    env_binary = os.getenv("FFMPEG_BINARY", "").strip()\n    candidates = [env_binary] if env_binary else []\n    path_binary = shutil.which("ffmpeg")\n    if path_binary:\n        candidates.append(path_binary)\n    if importlib.util.find_spec("imageio_ffmpeg"):\n        import imageio_ffmpeg\n\n        candidates.append(imageio_ffmpeg.get_ffmpeg_exe())\n    for candidate in candidates:\n        if candidate and Path(candidate).exists():\n            return candidate\n    return "ffmpeg"\n\n\ndef read_autoshot_frames(video_path: str, ffmpeg_executable: str):\n    require_module("ffmpeg", "pip install ffmpeg-python imageio-ffmpeg")\n    require_module("numpy", "pip install numpy")\n    import ffmpeg\n    import numpy as np\n\n    width, height = 48, 27\n    try:\n        video_stream, _ = (\n            ffmpeg.input(video_path)\n            .output("pipe:", format="rawvideo", pix_fmt="rgb24", s=f"{width}x{height}")\n            .run(cmd=ffmpeg_executable, capture_stdout=True, capture_stderr=True)\n        )\n    except ffmpeg.Error as exc:\n        stderr = exc.stderr.decode("utf-8", errors="replace") if exc.stderr else str(exc)\n        raise RuntimeError(f"FFmpeg failed for {video_path}. stderr: {stderr[-1200:]}") from exc\n    return np.frombuffer(video_stream, np.uint8).reshape([-1, height, width, 3])\n\n\ndef predict_boundary_scores(model: Any, video_path: str, repo_dir: Path, device: str):\n    require_module("numpy", "pip install numpy")\n    require_module("torch", "Kaggle GPU notebooks normally include torch.")\n    import numpy as np\n    import torch\n\n    add_repo_to_path(repo_dir)\n    from utils import get_batches\n\n    frames = read_autoshot_frames(video_path, resolve_ffmpeg_executable())\n    if len(frames) == 0:\n        raise RuntimeError(f"AutoShot could not read frames from video: {video_path}")\n\n    scores = []\n    with torch.no_grad():\n        for batch in get_batches(frames):\n            x = batch.transpose((3, 0, 1, 2))\n            x = x[np.newaxis, ...]\n            x = torch.from_numpy(x).float().to(device)\n            output = model(x)\n            logits = output[0] if isinstance(output, tuple) else output\n            prob = torch.sigmoid(logits[0]).detach().cpu().numpy()\n            scores.append(np.squeeze(prob)[25:75])\n    return np.concatenate(scores, axis=0)[: len(frames)]\n\n\ndef boundaries_to_shots(boundary_frames: Iterable[int], num_frames: int, min_shot_len: int) -> list[tuple[int, int]]:\n    boundaries = sorted(set(int(value) for value in boundary_frames))\n    shots: list[tuple[int, int]] = []\n    start = 0\n    for boundary in boundaries:\n        boundary = max(0, min(boundary, num_frames - 1))\n        end = boundary\n        if end - start + 1 >= min_shot_len:\n            shots.append((start, end))\n        start = boundary + 1\n    if start <= num_frames - 1:\n        end = num_frames - 1\n        if end - start + 1 >= min_shot_len:\n            shots.append((start, end))\n    return shots or [(0, num_frames - 1)]\n\n\ndef save_frame_at(cap: Any, frame_idx: int, out_path: Path, jpeg_quality: int) -> bool:\n    import cv2\n\n    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)\n    ok, frame_bgr = cap.read()\n    if not ok or frame_bgr is None:\n        return False\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    return bool(cv2.imwrite(str(out_path), frame_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), int(jpeg_quality)]))\n\n\ndef extract_representative_frames(\n    video_path: Path,\n    video_id: str,\n    shots: list[tuple[int, int]],\n    frames_dir: Path,\n    jpeg_quality: int,\n) -> list[dict[str, Any]]:\n    require_module("cv2", "pip install opencv-python-headless")\n    import cv2\n\n    cap = cv2.VideoCapture(str(video_path))\n    if not cap.isOpened():\n        raise RuntimeError(f"OpenCV cannot open video: {video_path}")\n\n    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)\n    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)\n    rows: list[dict[str, Any]] = []\n\n    for shot_index, (start, end) in enumerate(shots):\n        frame_items = [("first", start), ("middle", (start + end) // 2), ("last", end)]\n        for frame_type, frame_idx in frame_items:\n            filename = f"shot_{shot_index:04d}_{frame_type}_f{frame_idx:06d}.jpg"\n            local_path = frames_dir / video_id / filename\n            saved = save_frame_at(cap, frame_idx, local_path, jpeg_quality)\n            rows.append(\n                {\n                    "shot_id_local": shot_index,\n                    "shot_start_frame": start,\n                    "shot_end_frame": end,\n                    "shot_start_sec": start / fps if fps > 0 else "",\n                    "shot_end_sec": end / fps if fps > 0 else "",\n                    "frame_type": frame_type,\n                    "frame_idx": frame_idx,\n                    "frame_sec": frame_idx / fps if fps > 0 else "",\n                    "local_image_path": str(local_path),\n                    "saved": saved,\n                    "fps": fps,\n                    "total_frames_opencv": total_frames,\n                }\n            )\n    cap.release()\n    return rows\n\n\ndef resolve_checkpoint(cfg: Any, client: Any | None, layout: RunLayout) -> Path:\n    checkpoint = str(cfg_value(cfg, "CHECKPOINT_PATH", "") or "").strip()\n    if not checkpoint:\n        raise RuntimeError("Set CHECKPOINT_PATH to a local Kaggle path or gs:// checkpoint.")\n    if checkpoint.startswith("gs://"):\n        if client is None:\n            raise RuntimeError("A GCS client is required to download gs:// checkpoint.")\n        parsed = parse_gcs_uri(checkpoint)\n        local_path = layout.run_dir / "models" / PurePosixPath(parsed.blob_name).name\n        local_path.parent.mkdir(parents=True, exist_ok=True)\n        client.bucket(parsed.bucket).blob(parsed.blob_name).download_to_filename(str(local_path), timeout=900)\n        return local_path\n    path = Path(checkpoint)\n    if not path.exists():\n        raise FileNotFoundError(f"Checkpoint not found: {path}")\n    return path\n\n\ndef upload_file(bucket: Any, local_path: Path, object_key: str, content_type: str, overwrite: bool = True) -> None:\n    blob = bucket.blob(object_key)\n    blob.upload_from_filename(str(local_path), content_type=content_type, timeout=600)\n\n\ndef upload_text(bucket: Any, text: str, object_key: str, content_type: str = "text/plain") -> None:\n    bucket.blob(object_key).upload_from_string(text, content_type=content_type, timeout=600)\n\n\ndef upload_keyframe(bucket: Any, local_path: Path, object_key: str, skip_existing: bool, overwrite: bool) -> bool:\n    blob = bucket.blob(object_key)\n    if skip_existing and blob.exists():\n        return True\n    kwargs: dict[str, Any] = {"content_type": "image/jpeg", "timeout": 600}\n    if not overwrite:\n        kwargs["if_generation_match"] = 0\n    blob.upload_from_filename(str(local_path), **kwargs)\n    return True\n\n\ndef process_video_record(\n    cfg: Any,\n    record: dict[str, Any],\n    model: Any,\n    repo_dir: Path,\n    device: str,\n    layout: RunLayout,\n    output_bucket: Any | None,\n    upload_to_gcs: bool,\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\n    import numpy as np\n\n    started = time.perf_counter()\n    video_id = str(record["video_id"])\n    local_video = Path(str(record["local_video_path"]))\n    output_bucket_name = str(record.get("output_bucket") or "")\n    output_prefix = str(record["output_prefix"]).rstrip("/") + "/"\n    jpeg_quality = int(cfg_value(cfg, "JPEG_QUALITY", 95))\n    skip_existing = bool(cfg_value(cfg, "SKIP_EXISTING", True))\n    overwrite = bool(cfg_value(cfg, "OVERWRITE", False))\n\n    score_started = time.perf_counter()\n    scores = predict_boundary_scores(model, str(local_video), repo_dir, device)\n    score_ms = int((time.perf_counter() - score_started) * 1000)\n\n    boundary_frames = np.where(scores > float(record.get("threshold", cfg_value(cfg, "THRESHOLD", 0.296))))[0]\n    shots = boundaries_to_shots(boundary_frames, len(scores), int(record.get("min_shot_len", cfg_value(cfg, "MIN_SHOT_LEN", 5))))\n\n    frame_started = time.perf_counter()\n    frame_rows = extract_representative_frames(local_video, video_id, shots, layout.frames_dir, jpeg_quality)\n    frame_extract_ms = int((time.perf_counter() - frame_started) * 1000)\n\n    final_rows: list[dict[str, Any]] = []\n    upload_ms = 0\n    uploaded_count = 0\n    skipped_or_present_count = 0\n\n    for row in frame_rows:\n        filename = Path(str(row["local_image_path"])).name\n        image_storage_key = output_prefix + filename\n        image_gcs_uri = f"gs://{output_bucket_name}/{image_storage_key}" if output_bucket_name else ""\n        local_image_path = Path(str(row["local_image_path"]))\n\n        uploaded = False\n        if bool(row["saved"]):\n            if upload_to_gcs:\n                if output_bucket is None:\n                    raise RuntimeError("output_bucket is required when upload_to_gcs=True")\n                single_upload_started = time.perf_counter()\n                uploaded = upload_keyframe(output_bucket, local_image_path, image_storage_key, skip_existing, overwrite)\n                upload_ms += int((time.perf_counter() - single_upload_started) * 1000)\n                if uploaded:\n                    uploaded_count += 1\n            else:\n                uploaded = True\n                skipped_or_present_count += 1\n\n        shot_index = int(row["shot_id_local"])\n        frame_idx = int(row["frame_idx"])\n        final_rows.append(\n            {\n                "dataset_id": record["dataset_id"],\n                "batch_id": record["batch_id"],\n                "video_id": video_id,\n                "video_name": record["video_name"],\n                "video_gcs_uri": record["input_gcs_uri"],\n                "video_gcs_generation": record.get("input_generation", ""),\n                "shot_id": f"{video_id}_S{shot_index:04d}",\n                "shot_id_local": shot_index,\n                "shot_start_frame": row["shot_start_frame"],\n                "shot_end_frame": row["shot_end_frame"],\n                "shot_start_sec": row["shot_start_sec"],\n                "shot_end_sec": row["shot_end_sec"],\n                "frame_type": row["frame_type"],\n                "frame_idx": frame_idx,\n                "frame_sec": row["frame_sec"],\n                "keyframe_id": f"{video_id}_F{frame_idx:06d}",\n                "image_rel_path": f"{video_id}/{filename}",\n                "image_gcs_uri": image_gcs_uri,\n                "image_storage_key": image_storage_key,\n                "boundary_threshold": record.get("threshold", cfg_value(cfg, "THRESHOLD", 0.296)),\n                "min_shot_len": record.get("min_shot_len", cfg_value(cfg, "MIN_SHOT_LEN", 5)),\n                "saved": bool(row["saved"]) and uploaded,\n                "fps": row["fps"],\n                "total_frames_opencv": row["total_frames_opencv"],\n                "profile_version": record["profile_version"],\n                "run_id": record["run_id"],\n            }\n        )\n\n    frames_manifest_path = layout.artifacts_dir / "frames_manifest" / f"{video_id}.jsonl"\n    write_jsonl(frames_manifest_path, final_rows)\n    if upload_to_gcs and output_bucket is not None:\n        upload_file(output_bucket, frames_manifest_path, output_prefix + "frames_manifest.jsonl", "application/jsonl")\n\n    duration_ms = int((time.perf_counter() - started) * 1000)\n    summary = {\n        "run_id": record["run_id"],\n        "dataset_id": record["dataset_id"],\n        "batch_id": record["batch_id"],\n        "video_id": video_id,\n        "status": "success",\n        "input_path": str(local_video),\n        "input_gcs_uri": record["input_gcs_uri"],\n        "num_frames": int(len(scores)),\n        "num_boundaries": int(len(boundary_frames)),\n        "num_shots": int(len(shots)),\n        "num_keyframes": int(len(final_rows)),\n        "uploaded_keyframes": uploaded_count,\n        "local_or_skipped_keyframes": skipped_or_present_count,\n        "score_ms": score_ms,\n        "frame_extract_ms": frame_extract_ms,\n        "upload_ms": upload_ms,\n        "duration_ms": duration_ms,\n        "finished_at": utc_now_iso(),\n    }\n\n    if upload_to_gcs and bool(cfg_value(cfg, "CLEANUP_LOCAL_IMAGES_AFTER_UPLOAD", True)):\n        shutil.rmtree(layout.frames_dir / video_id, ignore_errors=True)\n\n    return final_rows, summary\n\n\ndef upload_run_artifacts(bucket: Any, layout: RunLayout, success: bool) -> None:\n    upload_file(bucket, layout.manifest_path, layout.gcs_artifact_prefix + "processing_manifest.jsonl", "application/jsonl")\n    upload_file(bucket, layout.shot_segments_path, layout.gcs_artifact_prefix + "shot_segments.csv", "text/csv")\n    upload_file(bucket, layout.errors_path, layout.gcs_artifact_prefix + "errors.jsonl", "application/jsonl")\n    upload_file(bucket, layout.video_summaries_path, layout.gcs_artifact_prefix + "video_summaries.jsonl", "application/jsonl")\n    upload_file(bucket, layout.summary_path, layout.gcs_artifact_prefix + "summary.json", "application/json")\n    upload_file(bucket, layout.log_path, layout.gcs_artifact_prefix + "run.log", "text/plain")\n    if success:\n        upload_text(bucket, "", layout.gcs_artifact_prefix + "_SUCCESS")\n\n\ndef run_pipeline(\n    cfg: Any,\n    run_kind: str,\n    batches: Any,\n    max_videos: int | None,\n    dry_run: bool,\n    upload_to_gcs: bool | None = None,\n) -> dict[str, Any]:\n    started_at = utc_now_iso()\n    started = time.perf_counter()\n    expected = [str(item).upper() for item in cfg_value(cfg, "EXPECTED_BATCHES", [])]\n    selected_batches = parse_selected_batches(batches, expected)\n    artifact_batch_id = selected_batches[0] if len(selected_batches) == 1 else "all"\n    run_id = new_run_id(run_kind)\n    layout = make_run_layout(cfg, run_id, artifact_batch_id)\n    logger = setup_logging(layout, bool(cfg_value(cfg, "VERBOSE", False)))\n\n    upload_enabled = bool(cfg_value(cfg, "UPLOAD_TO_GCS", True)) if upload_to_gcs is None else bool(upload_to_gcs)\n    bucket_name = resolve_bucket_name(cfg, require=upload_enabled and not dry_run)\n    client = make_storage_client(cfg) if upload_enabled and not dry_run else None\n    bucket = client.bucket(bucket_name) if client is not None and bucket_name else None\n\n    records, unmapped, input_root = discover_videos(cfg, selected_batches, max_videos, bucket_name)\n    for record in records:\n        record["run_id"] = run_id\n\n    write_jsonl(layout.manifest_path, records)\n    write_jsonl(layout.errors_path, [])\n    write_jsonl(layout.video_summaries_path, [])\n    init_csv(layout.shot_segments_path, SHOT_SEGMENTS_COLUMNS)\n\n    logger.info(\n        "run_id=%s kind=%s input_root=%s batches=%s dry_run=%s upload=%s planned_videos=%d unmapped=%d",\n        run_id,\n        run_kind,\n        input_root,\n        ",".join(selected_batches),\n        dry_run,\n        upload_enabled and not dry_run,\n        len(records),\n        len(unmapped),\n    )\n\n    if unmapped:\n        write_jsonl(layout.artifacts_dir / "unmapped.jsonl", unmapped)\n        logger.warning("wrote unmapped report with %d file(s)", len(unmapped))\n\n    if dry_run:\n        elapsed_ms = int((time.perf_counter() - started) * 1000)\n        summary = {\n            "run_id": run_id,\n            "run_kind": run_kind,\n            "stage": "dry_run",\n            "status": "success",\n            "dataset_id": str(cfg_value(cfg, "DATASET_ID", "ai_challenge_2025")),\n            "batches": selected_batches,\n            "input_root": str(input_root),\n            "planned_videos": len(records),\n            "unmapped_videos": len(unmapped),\n            "max_videos": max_videos,\n            "duration_ms": elapsed_ms,\n            "started_at": started_at,\n            "finished_at": utc_now_iso(),\n            "local_run_dir": str(layout.run_dir),\n            "gcs_artifact_prefix": layout.gcs_artifact_prefix,\n            "sample_record": records[0] if records else {},\n        }\n        write_json(layout.summary_path, summary)\n        logger.info("dry run finished planned_videos=%d duration_ms=%d", len(records), elapsed_ms)\n        return summary\n\n    if not records:\n        raise RuntimeError("No videos found. Check INPUT_ROOT, BATCHES, EXPECTED_BATCHES, and BATCH_REGEX.")\n\n    require_module("cv2", "pip install opencv-python-headless")\n    require_module("numpy", "pip install numpy")\n    require_module("torch", "Kaggle GPU notebooks normally include torch.")\n    require_module("ffmpeg", "pip install ffmpeg-python")\n    require_module("imageio_ffmpeg", "pip install imageio-ffmpeg")\n\n    repo_dir = ensure_autoshot_repo(cfg, logger)\n    checkpoint_path = resolve_checkpoint(cfg, client, layout)\n    device = select_device(str(cfg_value(cfg, "DEVICE", "auto")))\n    model = load_autoshot_model(repo_dir, checkpoint_path, device, logger)\n\n    total = len(records)\n    succeeded = 0\n    failed = 0\n    shot_rows_count = 0\n    keyframes_count = 0\n\n    progress = None\n    if bool(cfg_value(cfg, "USE_TQDM", True)):\n        try:\n            from tqdm.auto import tqdm\n\n            progress = tqdm(total=total, unit="video")\n        except Exception:\n            progress = None\n\n    for index, record in enumerate(records, 1):\n        pct_start = ((index - 1) / total) * 100\n        if bool(cfg_value(cfg, "LOG_EVERY_VIDEO", True)):\n            logger.info(\n                "[%d/%d %.1f%%] start video_id=%s batch=%s path=%s",\n                index,\n                total,\n                pct_start,\n                record["video_id"],\n                record["batch_id"],\n                record["local_video_path"],\n            )\n\n        try:\n            rows, video_summary = process_video_record(\n                cfg=cfg,\n                record=record,\n                model=model,\n                repo_dir=repo_dir,\n                device=device,\n                layout=layout,\n                output_bucket=bucket,\n                upload_to_gcs=upload_enabled,\n            )\n            append_csv_rows(layout.shot_segments_path, rows, SHOT_SEGMENTS_COLUMNS)\n            append_jsonl(layout.video_summaries_path, video_summary)\n            succeeded += 1\n            shot_rows_count += len(rows)\n            keyframes_count += int(video_summary.get("num_keyframes", 0))\n            pct_done = (index / total) * 100\n            logger.info(\n                "[%d/%d %.1f%%] done video_id=%s shots=%d keyframes=%d upload_ms=%d total_ms=%d",\n                index,\n                total,\n                pct_done,\n                record["video_id"],\n                video_summary.get("num_shots", 0),\n                video_summary.get("num_keyframes", 0),\n                video_summary.get("upload_ms", 0),\n                video_summary.get("duration_ms", 0),\n            )\n        except Exception as exc:  # noqa: BLE001 - keep processing remaining videos.\n            failed += 1\n            error_row = {\n                "run_id": run_id,\n                "dataset_id": record.get("dataset_id", ""),\n                "batch_id": record.get("batch_id", ""),\n                "video_id": record.get("video_id", ""),\n                "input_path": record.get("local_video_path", ""),\n                "input_gcs_uri": record.get("input_gcs_uri", ""),\n                "stage": "extract_upload",\n                "error_code": exc.__class__.__name__,\n                "error_message": str(exc),\n                "failed_at": utc_now_iso(),\n            }\n            append_jsonl(layout.errors_path, error_row)\n            logger.exception("[%d/%d] failed video_id=%s", index, total, record.get("video_id", ""))\n        finally:\n            if progress:\n                progress.set_postfix(succeeded=succeeded, failed=failed, refresh=False)\n                progress.update(1)\n\n    if progress:\n        progress.close()\n\n    elapsed_ms = int((time.perf_counter() - started) * 1000)\n    elapsed_sec = elapsed_ms / 1000 if elapsed_ms else 0\n    videos_per_hour = (succeeded / elapsed_sec * 3600) if elapsed_sec else 0\n    keyframes_per_min = (keyframes_count / elapsed_sec * 60) if elapsed_sec else 0\n    success = failed == 0 and succeeded == total and shot_rows_count > 0\n\n    summary = {\n        "run_id": run_id,\n        "run_kind": run_kind,\n        "stage": "extract_upload",\n        "status": "success" if success else "failed",\n        "dataset_id": str(cfg_value(cfg, "DATASET_ID", "ai_challenge_2025")),\n        "profile_version": str(cfg_value(cfg, "PROFILE_VERSION", "autoshot_v1")),\n        "batches": selected_batches,\n        "input_root": str(input_root),\n        "planned_videos": total,\n        "succeeded_videos": succeeded,\n        "failed_videos": failed,\n        "shot_rows": shot_rows_count,\n        "keyframes": keyframes_count,\n        "duration_ms": elapsed_ms,\n        "videos_per_hour": videos_per_hour,\n        "keyframes_per_min": keyframes_per_min,\n        "started_at": started_at,\n        "finished_at": utc_now_iso(),\n        "local_run_dir": str(layout.run_dir),\n        "gcs_artifact_prefix": layout.gcs_artifact_prefix,\n    }\n    write_json(layout.summary_path, summary)\n\n    if upload_enabled and bool(cfg_value(cfg, "UPLOAD_RUN_ARTIFACTS", True)) and bucket is not None:\n        summary["uploaded_run_artifacts"] = True\n        write_json(layout.summary_path, summary)\n        upload_run_artifacts(bucket, layout, success)\n        logger.info("uploaded run artifacts to gs://%s/%s", bucket_name, layout.gcs_artifact_prefix)\n\n    logger.info(\n        "run finished status=%s videos=%d/%d keyframes=%d duration_ms=%d videos_per_hour=%.2f",\n        summary["status"],\n        succeeded,\n        total,\n        keyframes_count,\n        elapsed_ms,\n        videos_per_hour,\n    )\n    return summary\n\n\ndef run_dry_run(cfg: Any) -> dict[str, Any]:\n    return run_pipeline(\n        cfg=cfg,\n        run_kind="dry_run",\n        batches=cfg_value(cfg, "DRY_RUN_BATCHES", "all"),\n        max_videos=cfg_value(cfg, "DRY_RUN_MAX_VIDEOS", 20),\n        dry_run=True,\n        upload_to_gcs=False,\n    )\n\n\ndef run_demo_one_batch(cfg: Any) -> dict[str, Any]:\n    return run_pipeline(\n        cfg=cfg,\n        run_kind="demo",\n        batches=cfg_value(cfg, "DEMO_BATCHES", "L21"),\n        max_videos=cfg_value(cfg, "DEMO_MAX_VIDEOS", 2),\n        dry_run=False,\n        upload_to_gcs=cfg_value(cfg, "UPLOAD_TO_GCS", True),\n    )\n\n\ndef run_full_dataset(cfg: Any) -> list[dict[str, Any]]:\n    expected = [str(item).upper() for item in cfg_value(cfg, "EXPECTED_BATCHES", [])]\n    batches = parse_selected_batches(cfg_value(cfg, "FULL_BATCHES", "all"), expected)\n    max_videos = cfg_value(cfg, "FULL_MAX_VIDEOS", None)\n    summaries: list[dict[str, Any]] = []\n    for batch_id in batches:\n        summaries.append(\n            run_pipeline(\n                cfg=cfg,\n                run_kind=f"full_{batch_id.lower()}",\n                batches=[batch_id],\n                max_videos=max_videos,\n                dry_run=False,\n                upload_to_gcs=cfg_value(cfg, "UPLOAD_TO_GCS", True),\n            )\n        )\n    return summaries\n\n\ndef preview_plan(cfg: Any, batches: Any | None = None, max_videos: int | None = 5) -> dict[str, Any]:\n    bucket_name = resolve_bucket_name(cfg, require=False)\n    selected = batches if batches is not None else cfg_value(cfg, "DRY_RUN_BATCHES", "all")\n    records, unmapped, input_root = discover_videos(cfg, selected, max_videos, bucket_name)\n    return {\n        "input_root": str(input_root),\n        "bucket": bucket_name,\n        "selected_batches": parse_selected_batches(selected, [str(item).upper() for item in cfg_value(cfg, "EXPECTED_BATCHES", [])]),\n        "sample_count": len(records),\n        "unmapped_sample_count": len(unmapped),\n        "sample_records": records[: min(len(records), max_videos or len(records))],\n    }\n'

if FORCE_REWRITE_HELPERS or not PARAMS_PATH.exists():
    PARAMS_PATH.write_text(PARAMS_TEMPLATE, encoding="utf-8")
if FORCE_REWRITE_HELPERS or not PIPELINE_PATH.exists():
    PIPELINE_PATH.write_text(PIPELINE_TEMPLATE, encoding="utf-8")

print(f"params: {PARAMS_PATH.resolve()}")
print(f"pipeline: {PIPELINE_PATH.resolve()}")
print("Edit video_to_frame_gcs_params.py before running demo/full cells.")


In [ ]:
%pip install -q google-cloud-storage ffmpeg-python imageio-ffmpeg einops opencv-python-headless tqdm


In [ ]:
import importlib
import video_to_frame_gcs_params as cfg
import video_to_frame_gcs_kaggle as pipe

importlib.reload(cfg)
importlib.reload(pipe)

pipe.preview_plan(cfg, max_videos=5)


## Dry Run

Discovers videos and writes local manifest artifacts only. It does not load AutoShot and does not upload to GCS.

In [ ]:
import importlib
import video_to_frame_gcs_params as cfg
import video_to_frame_gcs_kaggle as pipe

importlib.reload(cfg)
importlib.reload(pipe)

dry_summary = pipe.run_dry_run(cfg)
dry_summary


## Demo One Batch

Runs AutoShot on `DEMO_BATCHES` with `DEMO_MAX_VIDEOS`, uploads keyframes and run artifacts when `UPLOAD_TO_GCS=True`.

In [ ]:
import importlib
import video_to_frame_gcs_params as cfg
import video_to_frame_gcs_kaggle as pipe

importlib.reload(cfg)
importlib.reload(pipe)

demo_summary = pipe.run_demo_one_batch(cfg)
demo_summary


## Full Dataset

Before executing this cell, set `CONFIRM_FULL_RUN = "RUN_FULL_DATASET"` in `video_to_frame_gcs_params.py`. The helper runs one GCS artifact run per batch so downstream outputs stay grouped by batch.

In [ ]:
import importlib
import video_to_frame_gcs_params as cfg
import video_to_frame_gcs_kaggle as pipe

importlib.reload(cfg)
importlib.reload(pipe)

if getattr(cfg, "CONFIRM_FULL_RUN", "") != "RUN_FULL_DATASET":
    print('Skipped full run. Set CONFIRM_FULL_RUN = "RUN_FULL_DATASET" in video_to_frame_gcs_params.py to execute it.')
    full_summaries = []
else:
    full_summaries = pipe.run_full_dataset(cfg)

full_summaries


## Inspect Latest Local Artifacts

Use this after dry/demo/full cells to see summaries, CSVs, errors, and logs written under `RUN_DIR`.

In [ ]:
from pathlib import Path
import video_to_frame_gcs_params as cfg

run_root = Path(cfg.RUN_DIR)
latest = sorted([p for p in run_root.glob("*") if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)[:5]
for path in latest:
    print(path)
    for artifact in ["artifacts/summary.json", "artifacts/shot_segments.csv", "artifacts/errors.jsonl", "run.log"]:
        candidate = path / artifact
        if candidate.exists():
            print("  ", candidate, candidate.stat().st_size, "bytes")
